## 3 Omnilingual Evaluation

This notebook evaluates the finetuned Omnilingual model.

In [7]:
import os
import sys
import torch
torch.backends.cudnn.enabled = False
from tqdm import tqdm
from pathlib import Path
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

NOTEBOOK_DIR = Path.cwd()                             # scripts/
ROOT_DIR = NOTEBOOK_DIR.parent                        # omnilingual/
SUBMODULE_ROOT = ROOT_DIR / "omnilingual_asr"         # submodule root (contains workflows/)
sys.path.insert(0, str(SUBMODULE_ROOT))

from omnilingual_asr.utils import get_best_gpu, normalize_romansh_text, get_language_code_by_folder
from omnilingual_asr.constants import MODELS_ROOT

best_gpu = get_best_gpu()
if torch.cuda.is_available():
    os.environ["CUDA_VISIBLE_DEVICES"] = str(best_gpu)
    print(f"Using GPU {best_gpu}")
else:
    print("No GPU available – falling back to CPU")


Selected GPU 6 with 24088 MiB free memory
Using GPU 6


Now we can load the finetuned model weights from the checkpoint. Make sure to set `CHECKPOINT_FILE` to the path of your model's best checkpoint.

In [12]:
CHECKPOINT_FILE = MODELS_ROOT / "omnilingual-ctc-rm-1b-v2/ws_1.236d0922/checkpoints/step_30000/model/pp_00/tp_00/sdp_00.pt"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

# 1. Initialize the inference pipeline using the BASE model card. 
base_model_card = "omniASR_CTC_1B_v2"
print("Loading base model architecture...")
pipeline = ASRInferencePipeline(model_card=base_model_card, device=DEVICE)

from workflows.recipes.wav2vec2.asr.lora import LoraConfig, get_lora_model

# lora_config = LoraConfig(
#     r=8,
#     lora_alpha=16.0,
#     target_modules=["q_proj", "v_proj"],
#     lora_dropout=0.1,
# )
# print("Applying LoRA wrappers to the base model...")
# get_lora_model(pipeline.model, lora_config)

# 2. Load your fine-tuned checkpoint state dict
print("Loading fine-tuned weights...")
checkpoint = torch.load(CHECKPOINT_FILE, map_location="cpu")

# 3. Unwrap if needed (common keys: "model" or "module.")
state_dict = checkpoint.get("model", checkpoint)

# Remove "module." prefix if present (from DDP training)
state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

# 4. Inject the fine-tuned weights into the pipeline's instantiated model
# Now load the LoRA adapter weights
# checkpoint = torch.load(CHECKPOINT_FILE, map_location="cpu")
# state_dict = checkpoint.get("model", checkpoint)
# state_dict = {k.replace("module.", ""): v for k, v in state_dict.items()}

# Load only the LoRA parameters (strict=False ignores missing base weights)
# missing, unexpected = pipeline.model.load_state_dict(state_dict, strict=False)
pipeline.model.load_state_dict(state_dict, strict=False)
# for name, param in pipeline.model.named_parameters():
#     if 'lora_' in name:
#         param.data = param.data.to(torch.bfloat16)

pipeline.model.eval()  # Ensure model is in inference mode
print("LoRA adapters loaded successfully!")
print("Model weights loaded successfully!")

Device: cuda, Lang supported: True
Loading base model architecture...


/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/local/scratch/matuor/Romansh-ASR/omnilingual/.venv/lib/python3.11/site-packages/rich/live.py:231: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Loading fine-tuned weights...
LoRA adapters loaded successfully!
Model weights loaded successfully!


Then we can load the test data.

In [13]:
df_test = load_all_data("test")
df_test["sentence"] = df_test["sentence"].apply(normalize_romansh_text)
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


Then we can transcribe the test data.

In [14]:
audio_paths = df_test["audio_path"].tolist()
languages = df_test["idiom"].apply(get_language_code_by_folder).to_list()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch_paths = audio_paths[i:i+BATCH_SIZE]
    batch_langs = languages[i:i+BATCH_SIZE]
    try:
        # Transcribe the batch.
        results = pipeline.transcribe(
            batch_paths, 
            #lang=[LANGUAGE_CODE] * len(batch_paths), 
            lang = languages[i:i+BATCH_SIZE],
            batch_size=len(batch_paths)
        )
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at index {i}: {e}")
        transcriptions.extend([""] * len(batch_paths))

df_test["omnilingual_transcription"] = transcriptions

 54%|█████▍    | 43/79 [00:23<00:20,  1.79it/s]

Batch error at index 344: The map function has failed while processing the path 'data' of the input data. See nested exception for details.


100%|██████████| 79/79 [00:41<00:00,  1.89it/s]


Finally we can compute the metrics and show some transcription examples.

In [15]:
df_test["omnilingual_transcription"] = df_test["omnilingual_transcription"].apply(normalize_romansh_text)
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)

show_examples(df_test, "omnilingual_transcription")


OVERALL RESULTS
Total test samples: 631
Word Error Rate (WER): 14.68%
Character Error Rate (CER): 3.96%

PER IDIOM RESULTS

CC-2021-05-28
  Samples: 81
  WER: 5.01%
  CER: 1.46%

RMPUTER-CC-2021-06-11
  Samples: 114
  WER: 13.12%
  CER: 3.18%

RMSURSILV-CC-2021-05-28
  Samples: 94
  WER: 17.34%
  CER: 4.80%

RMSURSIV-CC-2021-12-23
  Samples: 151
  WER: 18.19%
  CER: 5.01%

RMSUTSILV-CC-2022-05-18
  Samples: 94
  WER: 17.57%
  CER: 4.55%

RMVALLADER-CC-2021-05-28
  Samples: 97
  WER: 14.06%
  CER: 4.00%

SUMMARY TABLE
                   idiom  samples  wer_mean  wer_std  cer_mean  cer_std
           cc-2021-05-28       81      5.01     4.64      1.46     1.86
   rmputer-cc-2021-06-11      114     13.12    11.12      3.18     3.05
 rmsursilv-cc-2021-05-28       94     17.34     9.73      4.80     3.77
  rmsursiv-cc-2021-12-23      151     18.19    18.33      5.01     6.54
 rmsutsilv-cc-2022-05-18       94     17.57     8.68      4.55     2.66
rmvallader-cc-2021-05-28       97     14.06 